# Phase 0 - Ingestion, Data Lake, Governance, DQ Gates, Metadata, RBAC

**Intended use:** Governed landing of hospital encounter data for analytics only (not a medical device).

**What this notebook does:** load every raw CSV registered in `datafile.txt`, write an immutable bronze snapshot, run data-quality gates, promote silver, and record manifest / metadata / RBAC artifacts.


## 1. Setup

Imports, project `ROOT`, and helpers that read/update `datafile.txt` so we never hardcode a single CSV path.


In [1]:
from __future__ import annotations
import json, hashlib, os, re, uuid, warnings, shutil
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
ROOT = Path(".").resolve()
if not (ROOT / "datafile.txt").exists():
    ROOT = Path("..").resolve()
os.chdir(ROOT)
DATAFILE = ROOT / "datafile.txt"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def load_registry(path=DATAFILE):
    rows = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split("|")
        if len(parts) < 4:
            continue
        rows.append({"role": parts[0].strip(), "zone": parts[1].strip(), "path": parts[2].strip(), "description": parts[3].strip()})
    return pd.DataFrame(rows)

def registry_paths(role=None):
    reg = load_registry()
    if role is not None:
        reg = reg[reg["role"] == role]
    return [ROOT / p for p in reg["path"].tolist()]

def registry_path(role, default=None):
    paths = registry_paths(role=role)
    if paths:
        return paths[0]
    return ROOT / default if default else None

def upsert_registry(role, zone, rel_path, description):
    lines = DATAFILE.read_text(encoding="utf-8").splitlines()
    new_line = f"{role}|{zone}|{rel_path}|{description}"
    out, found = [], False
    for line in lines:
        if line.startswith("#") or not line.strip():
            out.append(line)
            continue
        parts = line.split("|")
        if len(parts) >= 3 and parts[0].strip() == role and parts[2].strip() == rel_path:
            out.append(new_line)
            found = True
        else:
            out.append(line)
    if not found:
        out.append(new_line)
    DATAFILE.write_text("\n".join(out) + "\n", encoding="utf-8")

def new_run_id():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]

def file_checksum(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

print("ROOT:", ROOT)
print(load_registry())


ROOT: E:\Amit\Project\Hospital project
           role    zone                                       path  \
0           raw  bronze                 data/raw/diabetic_data.csv   
1         clean  silver        data/lake/silver/encounters.parquet   
2      features    gold      data/lake/gold/model_features.parquet   
3     sequences    gold       data/lake/gold/rnn_sequences.parquet   
4   predictions    gold    data/lake/gold/test_predictions.parquet   
5       metrics    gold  data/lake/gold/experiment_results.parquet   
6          mart  export          data/exports/mart_readmission.csv   
7          mart  export        data/exports/mart_clinical_risk.csv   
8          mart  export    data/exports/mart_model_performance.csv   
9          mart  export         data/exports/mart_dq_scorecard.csv   
10     manifest     ops    data/lake/bronze/_manifests/latest.json   
11     champion     ops              models/champion_register.json   

                                      description 

## 2. Create run ID

Every ingest gets a unique `run_id` for lineage and immutable bronze folders.


In [2]:
RUN_ID = new_run_id()
print('RUN_ID', RUN_ID)


RUN_ID 20260706T113515Z_6042d051


## 3. Load all raw sources from `datafile.txt`

Loops every `role=raw` entry. Add or remove CSVs in `datafile.txt` without changing this code.


In [3]:
raw_paths = registry_paths("raw")
assert raw_paths, "No raw entries in datafile.txt"
frames = []
ingest_details = []
for p in raw_paths:
    assert p.exists(), f"Missing raw file: {p}"
    df = pd.read_csv(p, low_memory=False)
    frames.append(df)
    ingest_details.append({
        "path": str(p.relative_to(ROOT)).replace("\\", "/"),
        "rows": int(len(df)),
        "cols": int(df.shape[1]),
        "checksum": file_checksum(p),
    })
    print(f"Loaded {p.name}: {df.shape}")

raw_df = pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]
print("Combined raw shape:", raw_df.shape)


Loaded diabetic_data.csv: (101766, 50)
Combined raw shape: (101766, 50)


## 4. Write bronze snapshot

One landing file in `data/lake/bronze/` (refreshed each run). Add more raw CSVs in `datafile.txt` if needed — you still get one combined bronze parquet here.


In [4]:
bronze_dir = ROOT / "data" / "lake" / "bronze"
bronze_dir.mkdir(parents=True, exist_ok=True)
bronze_path = bronze_dir / "encounters_raw.parquet"
raw_df.to_parquet(bronze_path, index=False)
# prune legacy per-run folders/manifests from older pipeline versions
for legacy in bronze_dir.iterdir():
    if legacy.is_dir() and legacy.name != "_manifests" and legacy.name[:1].isdigit():
        shutil.rmtree(legacy, ignore_errors=True)
manifest_dir = bronze_dir / "_manifests"
manifest_dir.mkdir(parents=True, exist_ok=True)
for legacy in manifest_dir.glob("*.json"):
    if legacy.name != "latest.json":
        legacy.unlink(missing_ok=True)
print("Bronze written:", bronze_path)


Bronze written: E:\Amit\Project\Hospital project\data\lake\bronze\encounters_raw.parquet


## 5. Data-quality checks (fail-fast)

Enterprise DQ dimensions: completeness, uniqueness, validity, consistency, timeliness. Critical failures stop the pipeline.


In [5]:
PLACEHOLDERS = {"?", "Unknown", "unknown", "-", ""}
dq = []

def add_dq(name, dimension, passed, detail, critical=True):
    dq.append({"check": name, "dimension": dimension, "passed": bool(passed), "detail": detail, "critical": critical})

add_dq("row_count_positive", "completeness", len(raw_df) > 0, f"rows={len(raw_df)}")
add_dq("has_encounter_id", "validity", "encounter_id" in raw_df.columns, "encounter_id present")
add_dq("has_readmitted", "validity", "readmitted" in raw_df.columns, "readmitted present")
if "encounter_id" in raw_df.columns:
    nuniq = raw_df["encounter_id"].nunique()
    add_dq("encounter_id_unique", "uniqueness", nuniq == len(raw_df), f"unique={nuniq} total={len(raw_df)}")
if "readmitted" in raw_df.columns:
    allowed = {"NO", "<30", ">30"}
    vals = set(raw_df["readmitted"].astype(str).unique())
    add_dq("readmitted_domain", "validity", vals.issubset(allowed), f"values={sorted(vals)}")
if "time_in_hospital" in raw_df.columns:
    los = pd.to_numeric(raw_df["time_in_hospital"], errors="coerce")
    add_dq("los_range", "consistency", ((los >= 1) & (los <= 14)).mean() > 0.95, f"mean_los={los.mean():.2f}")
if "gender" in raw_df.columns:
    g = set(raw_df["gender"].astype(str).unique())
    add_dq("gender_domain", "validity", g.issubset({"Female", "Male", "Unknown/Invalid"}), f"gender={g}")

placeholder_rate = {}
for c in raw_df.columns:
    s = raw_df[c].astype(str)
    placeholder_rate[c] = float(s.isin(PLACEHOLDERS).mean())
add_dq("ingest_timestamp", "timeliness", True, datetime.now(timezone.utc).isoformat())

dq_df = pd.DataFrame(dq)
critical_fail = dq_df[(~dq_df["passed"]) & (dq_df["critical"])]
print(dq_df)
if len(critical_fail):
    raise RuntimeError(f"DQ GATE FAILED: {critical_fail['check'].tolist()}")
print("DQ gate passed")


                 check     dimension  passed  \
0   row_count_positive  completeness    True   
1     has_encounter_id      validity    True   
2       has_readmitted      validity    True   
3  encounter_id_unique    uniqueness    True   
4    readmitted_domain      validity    True   
5            los_range   consistency    True   
6        gender_domain      validity    True   
7     ingest_timestamp    timeliness    True   

                                         detail  critical  
0                                   rows=101766      True  
1                          encounter_id present      True  
2                            readmitted present      True  
3                    unique=101766 total=101766      True  
4                   values=['<30', '>30', 'NO']      True  
5                                 mean_los=4.40      True  
6  gender={'Male', 'Female', 'Unknown/Invalid'}      True  
7              2026-07-06T11:35:16.800769+00:00      True  
DQ gate passed


## 6. Build silver layer

Replace placeholders with null, standardize text fields, write conformed parquet, update registry.


In [6]:
silver = raw_df.copy()
for c in silver.columns:
    if silver[c].dtype == object:
        silver[c] = silver[c].replace(list(PLACEHOLDERS), np.nan)

if "gender" in silver.columns:
    silver["gender"] = silver["gender"].astype(str).str.strip()
if "race" in silver.columns:
    silver["race"] = silver["race"].astype(str).str.strip()

silver_path = ROOT / "data" / "lake" / "silver" / "encounters.parquet"
silver_path.parent.mkdir(parents=True, exist_ok=True)
silver.to_parquet(silver_path, index=False)
upsert_registry("clean", "silver", "data/lake/silver/encounters.parquet", "Conformed encounters (silver)")
print("Silver written:", silver_path, silver.shape)


Silver written: E:\Amit\Project\Hospital project\data\lake\silver\encounters.parquet (101766, 50)


## 7. Export DQ scorecard

Certified export for ops / Power BI quality view.


In [7]:
dq_export = ROOT / "data" / "exports" / "mart_dq_scorecard.csv"
dq_export.parent.mkdir(parents=True, exist_ok=True)
dq_df.assign(run_id=RUN_ID).to_csv(dq_export, index=False)
upsert_registry("mart", "export", "data/exports/mart_dq_scorecard.csv", "DQ scorecard export")
print("DQ scorecard:", dq_export)


DQ scorecard: E:\Amit\Project\Hospital project\data\exports\mart_dq_scorecard.csv


## 8. Manifest, metadata catalog, and RBAC roles

Lineage manifest, dataset catalog (PII flags), and role definitions for later app enforcement.


In [8]:
manifest = {
    "run_id": RUN_ID,
    "ingested_at": datetime.now(timezone.utc).isoformat(),
    "sources": ingest_details,
    "bronze_path": str(bronze_path.relative_to(ROOT)).replace("\\", "/"),
    "silver_path": "data/lake/silver/encounters.parquet",
    "dq": dq_df.to_dict(orient="records"),
    "row_count": int(len(silver)),
}
manifest_path = ROOT / "data" / "lake" / "bronze" / "_manifests" / "latest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
upsert_registry("manifest", "ops", "data/lake/bronze/_manifests/latest.json", "Latest ingest run_id manifest")

nosql = ROOT / "data" / "nosql"
nosql.mkdir(parents=True, exist_ok=True)
(nosql / "pipeline_runs.json").write_text(json.dumps([manifest], indent=2), encoding="utf-8")

catalog = {
    "datasets": [
        {"name": "encounters_silver", "path": "data/lake/silver/encounters.parquet", "grain": "encounter", "pii_fields": ["encounter_id", "patient_nbr"]},
    ],
    "lineage": ["raw csv", "bronze", "silver", "modeling layer", "marts", "BI/app"],
    "run_id": RUN_ID,
}
(nosql / "metadata_catalog.json").write_text(json.dumps(catalog, indent=2), encoding="utf-8")

rbac = {
    "roles": {
        "admin": {"marts": "all", "predict": True, "ids": True, "audit": True},
        "analyst": {"marts": "all", "predict": True, "ids": True, "audit": False},
        "clinician": {"marts": ["mart_clinical_risk"], "predict": True, "ids": "limited", "audit": False},
        "viewer": {"marts": ["mart_readmission"], "predict": False, "ids": False, "audit": False},
    }
}
(nosql / "rbac_roles.json").write_text(json.dumps(rbac, indent=2), encoding="utf-8")
print("Manifest / catalog / RBAC written")


Manifest / catalog / RBAC written


## 9. Phase 0 summary


In [9]:
print("Phase 0 complete")
print("RUN_ID:", RUN_ID)
print("Silver rows:", len(silver))
print("DQ checks passed:", int(dq_df["passed"].sum()), "/", len(dq_df))


Phase 0 complete
RUN_ID: 20260706T113515Z_6042d051
Silver rows: 101766
DQ checks passed: 8 / 8
